# CustomGPT-QA — Architecture Walkthrough

This notebook explains the **math and code** of every piece of our decoder-only Transformer, going from a single embedding lookup all the way to autoregressive generation. The code blocks mirror the modules in `src/model/` exactly — we run small, shape-only examples so every operation is observable.

Roadmap:

1. **Token + positional embeddings** — turning integer IDs into vectors
2. **Scaled dot-product attention** — the core formula
3. **Causal masking** — why GPT is *causal*
4. **Multi-head attention** — running many heads in parallel
5. **Feed-forward + pre-LN block** — the residual recipe
6. **Putting it all together** — the full `GPT` module
7. **Generation** — autoregressive sampling
8. **Training objective** — cross-entropy on shifted targets
9. **Why pre-trained GPT-2 weights drop in** — the architectural payoff

We import torch once and reuse it everywhere.

In [1]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
torch.set_printoptions(precision=3, sci_mode=False, linewidth=120)

## 1. Token + positional embeddings

A language model sees a sequence of integer token IDs, e.g. `[15496, 11, 995, ...]`. To do any math we map each ID to a learned vector in $\mathbb{R}^{d}$ (here $d$ = `n_embd`).

Two parallel embeddings, **summed**:

$$
x_t \;=\; E_\text{tok}[\text{id}_t] \;+\; E_\text{pos}[t]
\qquad
E_\text{tok}\in\mathbb{R}^{V\times d}, \;\; E_\text{pos}\in\mathbb{R}^{T_\max\times d}
$$

* $E_\text{tok}$ — token embedding matrix, one row per vocab entry.
* $E_\text{pos}$ — **learned** positional embedding (GPT-2 style, not sinusoidal). Necessary because self-attention is order-invariant; without positions the model literally can't tell `"dog bites man"` apart from `"man bites dog"`.

This is exactly `src/model/embeddings.py`:

In [2]:
VOCAB, BLOCK, N_EMBD = 100, 8, 16  # tiny stand-ins for vocab_size, block_size, n_embd

tok_emb = nn.Embedding(VOCAB, N_EMBD)
pos_emb = nn.Embedding(BLOCK, N_EMBD)

idx = torch.tensor([[5, 17, 42, 3]])  # (B=1, T=4)
B, T = idx.shape
pos = torch.arange(T)                  # (T,) -> [0, 1, 2, 3]
x = tok_emb(idx) + pos_emb(pos)
print('idx shape:', tuple(idx.shape))
print('embedded x shape:', tuple(x.shape), '<- (B, T, n_embd)')
print('first vector:', x[0, 0, :5])

idx shape: (1, 4)
embedded x shape: (1, 4, 16) <- (B, T, n_embd)
first vector: tensor([ 0.146, -1.154, -2.533, -1.013,  0.396], grad_fn=<SliceBackward0>)


## 2. Scaled dot-product attention

Given a sequence of vectors $X\in\mathbb{R}^{T\times d}$ we project it into **queries**, **keys**, and **values**:

$$
Q = X W_Q,\quad K = X W_K,\quad V = X W_V
\qquad W_Q, W_K, W_V \in \mathbb{R}^{d\times d_k}
$$

The output is a weighted sum of values, with weights computed from similarity between queries and keys:

$$
\mathrm{Attention}(Q,K,V)
= \mathrm{softmax}\!\left(\frac{Q K^\top}{\sqrt{d_k}}\right) V
$$

Why the $\sqrt{d_k}$ scaling? For large $d_k$, the dot product $Q_i\!\cdot\! K_j$ has variance proportional to $d_k$. Without scaling, the softmax saturates (one value $\to 1$, the rest $\to 0$), gradients vanish, and training stalls. Dividing by $\sqrt{d_k}$ keeps the logits in a numerically healthy range — this is the *only* reason that constant is there.

Let's compute it by hand and confirm it matches a one-head version:

In [3]:
d_k = N_EMBD
W_q = nn.Linear(N_EMBD, d_k, bias=False)
W_k = nn.Linear(N_EMBD, d_k, bias=False)
W_v = nn.Linear(N_EMBD, d_k, bias=False)

Q = W_q(x)  # (B, T, d_k)
K = W_k(x)
V = W_v(x)

# scores[b, i, j] = Q_i . K_j / sqrt(d_k)
scores = (Q @ K.transpose(-2, -1)) / math.sqrt(d_k)
attn = F.softmax(scores, dim=-1)
out = attn @ V

print('Q, K, V shapes:', tuple(Q.shape), tuple(K.shape), tuple(V.shape))
print('attention weights (row sums to 1):')
print(attn[0])
print('attention output shape:', tuple(out.shape))

Q, K, V shapes: (1, 4, 16) (1, 4, 16) (1, 4, 16)
attention weights (row sums to 1):
tensor([[0.181, 0.190, 0.311, 0.318],
        [0.188, 0.255, 0.225, 0.332],
        [0.108, 0.239, 0.192, 0.462],
        [0.204, 0.226, 0.156, 0.415]], grad_fn=<SelectBackward0>)
attention output shape: (1, 4, 16)


Each row of `attn[0]` sums to 1 — it's a probability distribution over which past positions to attend to. Right now every position can see every other position, which is fine for an encoder but **wrong for a decoder LM**: at training time, position $t$ must not look at $t+1, t+2, \ldots$ — those are the future tokens we're trying to predict.

## 3. Causal masking

We zero out future positions by adding $-\infty$ to their scores **before** the softmax, so $e^{-\infty}=0$:

$$
\tilde{S}_{i,j} \;=\; \begin{cases} S_{i,j} & j \le i \\ -\infty & j > i \end{cases}
$$

We pre-compute the lower-triangular mask once and register it as a buffer (moves with `.to(device)` but isn't a learnable parameter). That's `self.register_buffer("causal_mask", ...)` in `src/model/attention.py`.

In [4]:
mask = torch.tril(torch.ones(T, T))  # 1 where j <= i, 0 elsewhere
print('causal mask (1 = allowed, 0 = future-token):')
print(mask)

masked_scores = scores.masked_fill(mask == 0, float('-inf'))
causal_attn = F.softmax(masked_scores, dim=-1)
print('\ncausal attention weights:')
print(causal_attn[0])
print('\nrow 0 only attends to col 0; row T-1 attends to all cols 0..T-1.')

causal mask (1 = allowed, 0 = future-token):
tensor([[1., 0., 0., 0.],
        [1., 1., 0., 0.],
        [1., 1., 1., 0.],
        [1., 1., 1., 1.]])

causal attention weights:
tensor([[1.000, 0.000, 0.000, 0.000],
        [0.424, 0.576, 0.000, 0.000],
        [0.200, 0.444, 0.356, 0.000],
        [0.204, 0.226, 0.156, 0.415]], grad_fn=<SelectBackward0>)

row 0 only attends to col 0; row T-1 attends to all cols 0..T-1.


Notice row 0 is `[1, 0, 0, 0]` — position 0 has no past, so 100% attention on itself. Row 3 spreads attention over all four positions. This *triangular* structure is what makes the model **autoregressive**: predicting token $t+1$ only ever depends on tokens $0, \ldots, t$.

## 4. Multi-head attention

One attention head can only blend information one way. Multi-head attention runs $h$ heads **in parallel** with smaller per-head dimension $d_k = d / h$, then concatenates and projects:

$$
\mathrm{MultiHead}(X) = \mathrm{Concat}(\text{head}_1, \ldots, \text{head}_h)\, W_O
$$

Why? Each head can specialize: one might attend to the immediately previous token (local n-gram), another might attend to subjects of verbs, etc. Empirically, splitting the embedding into independent heads outperforms one giant head with the same total parameter count.

**Implementation trick.** Instead of three separate `Linear` layers for $W_Q, W_K, W_V$, we do one fused `Linear` of width $3d$ and split. The view/transpose dance gets each head into its own dimension so PyTorch batches them automatically:

In [5]:
n_head = 4
head_dim = N_EMBD // n_head        # = 4

qkv = nn.Linear(N_EMBD, 3 * N_EMBD, bias=False)
proj = nn.Linear(N_EMBD, N_EMBD, bias=False)

B, T, C = x.shape
q, k, v = qkv(x).split(N_EMBD, dim=2)     # each (B, T, C)
# (B, T, C) -> (B, n_head, T, head_dim)
q = q.view(B, T, n_head, head_dim).transpose(1, 2)
k = k.view(B, T, n_head, head_dim).transpose(1, 2)
v = v.view(B, T, n_head, head_dim).transpose(1, 2)

scores = (q @ k.transpose(-2, -1)) / math.sqrt(head_dim)
scores = scores.masked_fill(mask == 0, float('-inf'))
attn = F.softmax(scores, dim=-1)
y = attn @ v                              # (B, n_head, T, head_dim)
y = y.transpose(1, 2).contiguous().view(B, T, C)  # re-assemble heads
y = proj(y)
print('multi-head output:', tuple(y.shape), '<- back to (B, T, n_embd)')

multi-head output: (1, 4, 16) <- back to (B, T, n_embd)


This is *exactly* what `MultiHeadMaskedSelfAttention.forward` does in `src/model/attention.py` (plus dropouts on the attention weights and on the residual projection, which we omit here for clarity).

## 5. Feed-forward network + pre-LN block

Each Transformer block has **two sub-layers**, both wrapped in residual connections:

$$
\begin{aligned}
x &\leftarrow x + \mathrm{Attn}(\mathrm{LN}(x)) \\
x &\leftarrow x + \mathrm{FFN}(\mathrm{LN}(x))
\end{aligned}
$$

The FFN is a small position-wise MLP that expands to $4d$ and back:

$$
\mathrm{FFN}(x) = W_2 \,\mathrm{GELU}(W_1 x + b_1) + b_2
\qquad W_1\in\mathbb{R}^{d\times 4d}, \;\; W_2\in\mathbb{R}^{4d\times d}
$$

Two important details we use in `src/model/block.py`:

* **Pre-LN, not post-LN.** Original Transformer had `LN(x + sublayer(x))`. The modern GPT-2 variant moves LayerNorm *inside* the residual (`x + sublayer(LN(x))`). Pre-LN is much more stable at depth — training a 12-layer post-LN model from scratch typically needs warmup tricks; pre-LN just trains.
* **GELU (tanh approximation)** — the smooth activation HuggingFace GPT-2 uses. Without `approximate='tanh'` our numbers would drift slightly from a pretrained GPT-2 init.

Residual connections add the input to the sub-layer output. This lets gradients flow directly from the loss back to early layers (the famous "residual highway") and lets the network learn small corrections to the identity rather than entire transformations from scratch.

In [6]:
class ToyFFN(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d, 4 * d),
            nn.GELU(approximate='tanh'),
            nn.Linear(4 * d, d),
        )
    def forward(self, x): return self.net(x)

ln1 = nn.LayerNorm(N_EMBD)
ln2 = nn.LayerNorm(N_EMBD)
ffn = ToyFFN(N_EMBD)

# A toy 'attention' replaced by identity for shape check
def toy_attn(z): return y  # reuse multi-head output from above

x_in = x.clone()
x_mid = x_in + toy_attn(ln1(x_in))      # residual around attention
x_out = x_mid + ffn(ln2(x_mid))          # residual around FFN
print('block output shape:', tuple(x_out.shape), '(unchanged from input — that\'s the point of residuals)')

block output shape: (1, 4, 16) (unchanged from input — that's the point of residuals)


## 6. The full model

Stack $N$ blocks, add a final LayerNorm, and project back to vocabulary logits:

$$
z = \mathrm{LN}(\text{Block}_N(\cdots \text{Block}_1(\text{Embed}(\text{idx})) \cdots))
\qquad
\text{logits} = z W_E^\top
$$

**Weight tying.** The output projection $W_E^\top$ shares parameters with the token embedding $E_\text{tok}$. This halves the parameter count of the output head and (empirically) improves generalization — the embedding for token *X* and the row of the LM head that predicts *X* describe the same concept, so why not tie them?

Let's instantiate the real model from `src/model/gpt.py` and look at param counts. We use a small override (4 layers, 4 heads) so the cell stays fast — the real defaults match GPT-2 small (12/12/768/1024).

In [7]:
import sys
sys.path.insert(0, '..')
from src.config import GPTConfig
from src.model.gpt import GPT

cfg = GPTConfig(n_layer=4, n_head=4, n_embd=128, block_size=64, dropout=0.0)
model = GPT(cfg)
print(f'model params: {sum(p.numel() for p in model.parameters())/1e6:.2f}M  (tiny, for the notebook)')
print(f'configurable shape: n_layer={cfg.n_layer}, n_head={cfg.n_head}, n_embd={cfg.n_embd}, block_size={cfg.block_size}')

# Confirm weight tying — the LM head weight IS the token-embedding matrix.
print('lm_head.weight is tok_emb.weight?', model.lm_head.weight is model.embed.tok_emb.weight)

ids = torch.randint(0, cfg.vocab_size, (2, 16))   # (B=2, T=16)
targets = torch.randint(0, cfg.vocab_size, (2, 16))
logits, loss = model(ids, targets)
print('\nlogits:', tuple(logits.shape), '<- (B, T, vocab)')
print(f'initial loss: {loss.item():.3f}  ~ ln(vocab) = {math.log(cfg.vocab_size):.3f}')

model params: 7.23M  (tiny, for the notebook)
configurable shape: n_layer=4, n_head=4, n_embd=128, block_size=64
lm_head.weight is tok_emb.weight? True



logits: (2, 16, 50257) <- (B, T, vocab)
initial loss: 10.821  ~ ln(vocab) = 10.825


The initial loss is very close to $\ln(V)$ — exactly what we expect from an untrained model: it assigns roughly uniform probability over the vocabulary, so cross-entropy is $\ln(V) \approx 10.82$ for $V=50257$. This is a one-line sanity check we use whenever we resize the model.

## 7. Generation

At inference time we feed the prompt, take the **last** position's logits, sample one token, append it, and repeat. The model is unchanged — the recurrence is in the sampling loop:

1. Crop input to last `block_size` tokens (we only have `block_size` learned positions).
2. Run forward; take `logits[:, -1, :]`.
3. Divide by `temperature` (smaller = sharper).
4. Optionally **top-$k$**: zero out everything below the $k$-th largest logit.
5. Softmax → `multinomial` to sample one token.
6. Append, repeat.

Temperature $\tau$ rescales logits by $1/\tau$. At $\tau=1$ we sample from the model's distribution as-is; at $\tau<1$ we sharpen toward the most-likely token (more deterministic); $\tau\to 0$ becomes greedy argmax.

In [8]:
prompt = torch.tensor([[10, 22, 33]])
with torch.no_grad():
    out = model.generate(prompt, max_new_tokens=5, temperature=1.0, top_k=20)
print('prompt:    ', prompt.tolist())
print('generated: ', out.tolist())
print('Note: this untrained model will produce nonsense IDs — the *mechanics* of generation are what we are showing here.')

prompt:     [[10, 22, 33]]
generated:  [[10, 22, 33, 32572, 29575, 46436, 7617, 16918]]
Note: this untrained model will produce nonsense IDs — the *mechanics* of generation are what we are showing here.


## 8. Training objective

We minimize **cross-entropy** between the predicted next-token distribution and the true next token. Concretely: given a sequence of token IDs $(s_0, s_1, \ldots, s_T)$, we feed inputs $x = (s_0, \ldots, s_{T-1})$ and targets $y = (s_1, \ldots, s_T)$ — i.e. the targets are the inputs *shifted by one*:

$$
\mathcal{L} = -\frac{1}{T}\sum_{t=1}^{T} \log p_\theta(s_t \mid s_0, \ldots, s_{t-1})
$$

This is one forward pass; the causal mask in attention is what guarantees the prediction at position $t$ only depends on positions $\le t-1$, so the entire loss can be computed in parallel. **Perplexity** is just $e^\mathcal{L}$ — the geometric-mean inverse probability the model assigns to each true token. Lower is better.

## 9. Why pre-trained GPT-2 weights drop in

Every module above — token + learned-position embeddings, fused-QKV attention with causal mask, multi-head split, 4× tanh-GELU FFN, pre-LN residual blocks, weight-tied LM head — **is** GPT-2. With the defaults in `src/config.py` (12 layer, 12 head, 768 emb, 1024 block, bias=True) our `state_dict` keys are 1-to-1 with HuggingFace's, modulo:

* HF stores attention QKV / MLP weights as `Conv1D`, which is `nn.Linear` with the weight matrix transposed. We `.t()` those four tensors per block when copying.
* HF's internal `attn.bias`/`attn.masked_bias` are the same triangular mask we register as `causal_mask`, so we skip them.

Everything else copies literally. See `src/load_gpt2.py` — 148/148 tensors copy cleanly for `gpt2-small`. After this load, the same model architecture that was getting val perplexity ~274 from scratch on our 370k-token corpus drops to ~29 zero-shot and ~26 after a short fine-tune. The architecture was already right; we just gave it priors learned on 40 GB of internet text.

---

That's the whole model. Every line in `src/model/` corresponds to one of the equations above; every CLI in `src/` (train, evaluate, rag, gui) is wired around this single `GPT` class.